In [1]:
import pandas as pd
import numpy as np

import json

In [7]:
df = pd.read_csv("../data/sample_pairs_scenario.csv")

In [8]:
distribution = df.groupby("scenario")["pair_id"].nunique()
print(distribution)

scenario
answer unknown             388
false premise              243
stale                       13
subjective                 127
underspecified context    2076
underspecified intent      719
Name: pair_id, dtype: int64


In [10]:
df = pd.read_csv("../data/sample_pairs_with_scenario.csv")
distribution = df.groupby("scenario")["pair_id"].nunique()
print(distribution)

scenario
answer unknown             387
false premise              243
stale                       13
subjective                 128
underspecified context    3435
underspecified intent      719
Name: pair_id, dtype: int64


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = [
    # Qwen 2.5 0.5B - Overall
    {'Model': 'Qwen 2.5 0.5B', 'Category': 'Overall', 'Experiment': 'Exp 1 (Base)', 'Precision': 52.17, 'Recall': 36.85, 'F1': 39.85},
    {'Model': 'Qwen 2.5 0.5B', 'Category': 'Overall', 'Experiment': 'Exp 2 (Weighted)', 'Precision': 55.44, 'Recall': 24.87, 'F1': 30.36},
    {'Model': 'Qwen 2.5 0.5B', 'Category': 'Overall', 'Experiment': 'Exp 2 (No Stale)', 'Precision': 56.82, 'Recall': 20.42, 'F1': 26.05},
    
    # Qwen 2.5 0.5B - Rulebreakers
    {'Model': 'Qwen 2.5 0.5B', 'Category': 'Rulebreakers', 'Experiment': 'Exp 1 (Base)', 'Precision': 49.98, 'Recall': 69.67, 'F1': 58.21},
    {'Model': 'Qwen 2.5 0.5B', 'Category': 'Rulebreakers', 'Experiment': 'Exp 2 (Weighted)', 'Precision': 49.76, 'Recall': 73.90, 'F1': 59.47},
    {'Model': 'Qwen 2.5 0.5B', 'Category': 'Rulebreakers', 'Experiment': 'Exp 2 (No Stale)', 'Precision': 49.75, 'Recall': 70.82, 'F1': 58.44},
    
    # Qwen 2.5 7B - Overall
    {'Model': 'Qwen 2.5 7B', 'Category': 'Overall', 'Experiment': 'Exp 1 (Base)', 'Precision': 65.10, 'Recall': 13.58, 'F1': 16.90},
    {'Model': 'Qwen 2.5 7B', 'Category': 'Overall', 'Experiment': 'Exp 2 (Weighted)', 'Precision': 60.36, 'Recall': 13.11, 'F1': 16.16},
    {'Model': 'Qwen 2.5 7B', 'Category': 'Overall', 'Experiment': 'Exp 2 (No Stale)', 'Precision': 65.33, 'Recall': 11.55, 'F1': 14.70},
    
    # Qwen 2.5 7B - Rulebreakers
    {'Model': 'Qwen 2.5 7B', 'Category': 'Rulebreakers', 'Experiment': 'Exp 1 (Base)', 'Precision': 52.38, 'Recall': 8.46, 'F1': 14.57},
    {'Model': 'Qwen 2.5 7B', 'Category': 'Rulebreakers', 'Experiment': 'Exp 2 (Weighted)', 'Precision': 52.23, 'Recall': 9.67, 'F1': 16.32},
    {'Model': 'Qwen 2.5 7B', 'Category': 'Rulebreakers', 'Experiment': 'Exp 2 (No Stale)', 'Precision': 44.06, 'Recall': 4.89, 'F1': 8.80},
]

df = pd.DataFrame(data)

sns.set_theme(style='whitegrid')
colors = {
    'Exp 1 (Base)': '#4285F4',       # Google Blue
    'Exp 2 (Weighted)': '#34A853',   # Google Green
    'Exp 2 (No Stale)': '#137333'    # Darker Green 
}

# bar charts
metrics = ['Precision', 'Recall', 'F1']

for metric in metrics:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
    fig.suptitle(f'{metric} Comparison Across 3 Experiments', fontsize=16, fontweight='bold')

    for i, category in enumerate(['Overall', 'Rulebreakers']):
        subset = df[df['Category'] == category]
        ax = axes[i]
        sns.barplot(data=subset, x='Model', y=metric, hue='Experiment', palette=colors, ax=ax)
        
        ax.set_title(f'{category} Dataset', fontsize=14)
        ax.set_ylabel(f'{metric} (%)' if i == 0 else '')
        ax.set_xlabel('')
        ax.set_ylim(0, 100) # Increased to 100% for standard percentage scaling
        
        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f%%', padding=3, size=9)

    plt.tight_layout()
    filename = f'{metric.lower()}_comparison_3way.png'
    plt.savefig(filename, dpi=300)
    plt.close(fig)
    print(f"Saved {metric} bar chart as '{filename}'")

# scatter
fig_scatter, ax_scatter = plt.subplots(figsize=(11, 8))

sns.scatterplot(
    data=df, x='Recall', y='Precision', 
    hue='Experiment', style='Model', 
    s=200, palette=colors, markers={'Qwen 2.5 0.5B': 'o', 'Qwen 2.5 7B': 's'}, ax=ax_scatter
)

for model in df['Model'].unique():
    for category in df['Category'].unique():
        # Filter down to the specific group
        base = df[(df['Model'] == model) & (df['Category'] == category) & (df['Experiment'] == 'Exp 1 (Base)')].iloc[0]
        weighted = df[(df['Model'] == model) & (df['Category'] == category) & (df['Experiment'] == 'Exp 2 (Weighted)')].iloc[0]
        no_stale = df[(df['Model'] == model) & (df['Category'] == category) & (df['Experiment'] == 'Exp 2 (No Stale)')].iloc[0]
        
        # Arrow: Base -> Weighted (Dashed Line)
        ax_scatter.annotate(
            '', xy=(weighted['Recall'], weighted['Precision']), 
            xytext=(base['Recall'], base['Precision']),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls='--')
        )
        
        # Arrow: Base -> No Stale (Dotted Line)
        ax_scatter.annotate(
            '', xy=(no_stale['Recall'], no_stale['Precision']), 
            xytext=(base['Recall'], base['Precision']),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls=':')
        )
        
        # Label the base point so you know which cluster is which
        ax_scatter.text(
            base['Recall'] + 1, base['Precision'] + 0.5, 
            f'{model}\n({category})', 
            fontsize=9, alpha=0.8
        )

ax_scatter.set_title('Precision vs. Recall Trade-off (Movement from Base to Exp 2 Variants)', fontsize=15, fontweight='bold', pad=15)
ax_scatter.set_xlabel('Recall (%)', fontsize=12)
ax_scatter.set_ylabel('Precision (%)', fontsize=12)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.tight_layout()
plt.savefig('precision_recall_scatter_3way.png', dpi=300)
plt.close(fig_scatter)
print("Saved Scatter Plot as 'precision_recall_scatter_3way.png'")

Saved Precision bar chart as 'precision_comparison_3way.png'
Saved Recall bar chart as 'recall_comparison_3way.png'
Saved F1 bar chart as 'f1_comparison_3way.png'
Saved Scatter Plot as 'precision_recall_scatter_3way.png'
